# 🧙‍♂️ VYAS-Daivagya-1.5B: Vedic Astrology Foundation Model Trainer
### ⚡ 100% Crash-Proof Fine-Tuning on Free Google Colab (T4 GPU)

> **⚠️ महत्वपूर्ण निर्देश (GPU Check):**
> ऊपर मेनू में जाएं: **Runtime** ➔ **Change runtime type** ➔ **Hardware accelerator: T4 GPU** चुनें और Save करें।

In [ ]:
# Step 1: Verify GPU & Install Production-Grade Fine-Tuning Stack (No Segfaults / Crash-Proof)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('⚠️ GPU सक्रिय नहीं है! कृपया Runtime -> Change runtime type -> T4 GPU चुनें और पुनः चलाएं।')

print(f'✅ GPU Detected: {torch.cuda.get_device_name(0)}')
print(f'✅ VRAM Available: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')

print('📦 Installing Hugging Face TRL, PEFT, Accelerate & BitsAndBytes...')
!pip install -q transformers peft trl bitsandbytes accelerate datasets
print('🎉 All dependencies installed successfully!')

In [ ]:
# Step 2: Load Base Model with 4-Bit NormalFloat Quantization (QLoRA)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
print(f'⏳ Loading {model_id} in 4-bit memory-efficient mode...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
)
print('✅ Base Model Loaded in 4-Bit!')

In [ ]:
# Step 3: Setup Low-Rank Adaptation (LoRA) Matrix
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
print('✅ LoRA Matrix Configured!')

In [ ]:
# Step 4: Load & Format VYAS-Daivagya Astrological SFT Dataset
import os
from datasets import load_dataset

dataset_file = 'vyas_daivagya_sft_train.jsonl'

# If not in /content, prompt for upload
if not os.path.exists(dataset_file):
    print('📥 dataset file not found in /content. Opening file upload prompt...')
    print('👉 Please select your vyas_daivagya_sft_train.jsonl file from your PC:')
    from google.colab import files
    files.upload()

dataset = load_dataset('json', data_files=dataset_file, split='train')

def format_chat_prompt(batch):
    texts = []
    for msgs in batch['messages']:
        formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        texts.append(formatted)
    return {'text': texts}

formatted_dataset = dataset.map(format_chat_prompt, batched=True)
print(f'✅ Successfully loaded and verified {len(formatted_dataset)} classical training samples!')

In [ ]:
# Step 5: Execute Training (Memory Safe: Batch=2, GradAccum=8, MaxLen=1024, No OOM crashes!)
from trl import SFTTrainer
from transformers import TrainingArguments

torch.cuda.empty_cache()

training_args = TrainingArguments(
    output_dir='vyas_daivagya_output',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    num_train_epochs=2,
    fp16=True,
    logging_steps=10,
    save_strategy='epoch',
    optim='paged_adamw_8bit',
    report_to='none',
    seed=42
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    dataset_text_field='text',
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args,
)

print('🚀 Starting VYAS-Daivagya Fine-Tuning Execution...')
trainer.train()
print('🎉 Training Completed Successfully!')

# Save trained LoRA weights
model.save_pretrained('vyas_daivagya_lora_model')
tokenizer.save_pretrained('vyas_daivagya_lora_model')
print('💾 LoRA Weights saved in ./vyas_daivagya_lora_model')

In [ ]:
# Step 6: Test Inference Instantly
test_prompt = [
    {'role': 'system', 'content': 'आप आचार्य व्यास हैं—महर्षि पराशर, जैमिनी, एवं भृगु ज्योतिष के मूर्धन्य वैदिक विद्वान।'},
    {'role': 'user', 'content': 'मेरी कुण्डली में सूर्य 7° वृश्चिक राशि में है, क्या यह पुष्कर नवांश में है? इसका क्या फल होगा?'}
]
text = tokenizer.apply_chat_template(test_prompt, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt').to('cuda')

outputs = model.generate(**inputs, max_new_tokens=400, temperature=0.3)
response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)

print('\n🧙‍♂️ आचार्य व्यास उत्तर:\n')
print(response)

In [ ]:
# Step 7: Auto-Backup Model to Google Drive & Download Package
try:
    from google.colab import drive
    print('💾 Mounting Google Drive to backup model safely...')
    drive.mount('/content/drive')
    import shutil
    shutil.make_archive('vyas_daivagya_lora_v1', 'zip', 'vyas_daivagya_lora_model')
    shutil.copy('vyas_daivagya_lora_v1.zip', '/content/drive/MyDrive/vyas_daivagya_lora_v1.zip')
    print('🎉 Model successfully backed up to Google Drive: MyDrive/vyas_daivagya_lora_v1.zip')
except Exception as e:
    print(f'Drive backup note: {e}. You can download vyas_daivagya_lora_model directly from the left sidebar.')